# Pipeline de ingenieria de datos: musica en Colombia

Este cuaderno desarrolla una simulacion completa de ingenieria de datos usando conjuntos abiertos relacionados con musica, cultura, conciertos, festivales y agentes musicales en Colombia. El flujo sigue el ciclo de vida solicitado: seleccion del tema, origen de datos, limpieza, movimiento entre sistemas, transformacion, resultados finales y reflexion.

**Tema elegido:** actividad musical en Colombia a partir del portal nacional de datos abiertos [datos.gov.co](https://www.datos.gov.co/).

**Pregunta guia:** que departamentos, municipios y tipos de actividad aparecen con mayor frecuencia en los datos publicos disponibles sobre musica en Colombia?

## 1. Obtencion y origen de los datos

La fuente principal es el catalogo Socrata de Datos Abiertos Colombia. El cuaderno consulta automaticamente el catalogo con palabras clave como `musica`, `música`, `eventos musicales` y `cultura musica`. Luego toma los recursos encontrados y descarga registros desde la API SODA.

Arquitectura de ingesta propuesta:

1. **Catalogo:** `https://api.us.socrata.com/api/catalog/v1` para descubrir datasets.
2. **API de datos:** `https://www.datos.gov.co/resource/{id}.json` para descargar registros.
3. **Zona raw:** archivo JSON/tabla SQLite con datos sin modificar.
4. **Zona limpia:** datos normalizados y deduplicados.
5. **Data mart:** tablas agregadas para dashboard.

In [ ]:
from pathlib import Path
import json
import re
import sqlite3
from datetime import datetime

import pandas as pd
import requests

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
WAREHOUSE_DIR = DATA_DIR / 'warehouse'

for folder in [RAW_DIR, PROCESSED_DIR, WAREHOUSE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CATALOG_URL = 'https://api.us.socrata.com/api/catalog/v1'
DOMAINS = 'www.datos.gov.co,datos.gov.co'
QUERIES = ['musica', 'música', 'eventos musicales', 'cultura musica']

print('Carpeta base:', BASE_DIR)

In [ ]:
def catalog_search(query: str, limit: int = 20) -> list[dict]:
    params = {
        'only': 'dataset',
        'domains': DOMAINS,
        'q': query,
        'limit': limit,
    }
    response = requests.get(CATALOG_URL, params=params, timeout=30)
    response.raise_for_status()
    return response.json().get('results', [])

datasets = {}
for query in QUERIES:
    for item in catalog_search(query):
        resource = item.get('resource', {})
        text = ' '.join([resource.get('name', ''), resource.get('description', ''), ' '.join(resource.get('tags', []))]).lower()
        if resource.get('id') and re.search(r'música|musica|musical|cultura|festival|concierto|artista|banda|orquesta', text):
            datasets[resource['id']] = resource

catalog_df = pd.DataFrame([
    {
        'id': rid,
        'nombre': res.get('name'),
        'descripcion': res.get('description'),
        'filas_aprox': res.get('columns_field_name'),
        'url': res.get('permalink'),
    }
    for rid, res in datasets.items()
])

catalog_df.head(10)

## 2. Ingesta y almacenamiento raw

La ingesta descarga hasta 5.000 registros del primer recurso que entregue datos. En un proyecto productivo se agregarian paginacion, token Socrata, reintentos, bitacora y validaciones de esquema. Para la actividad, el objetivo es demostrar el flujo completo con una fuente publica y reproducible.

In [ ]:
def download_resource(resource_id: str, limit: int = 5000) -> list[dict]:
    url = f'https://www.datos.gov.co/resource/{resource_id}.json'
    response = requests.get(url, params={'$limit': limit}, timeout=45)
    response.raise_for_status()
    return response.json()

raw_rows = []
selected_resource = None

for resource_id, resource in datasets.items():
    try:
        rows = download_resource(resource_id)
        if rows:
            raw_rows = rows
            selected_resource = resource
            break
    except Exception as exc:
        print('No se pudo descargar', resource_id, exc)

if not raw_rows:
    raw_rows = [
        {'nombre': 'Festival Petronio Alvarez', 'departamento': 'Valle del Cauca', 'municipio': 'Cali', 'tipo': 'Festival', 'fecha': '2025-08-15'},
        {'nombre': 'Rock al Parque', 'departamento': 'Bogota D.C.', 'municipio': 'Bogota', 'tipo': 'Festival', 'fecha': '2025-11-09'},
        {'nombre': 'Festival Vallenato', 'departamento': 'Cesar', 'municipio': 'Valledupar', 'tipo': 'Festival', 'fecha': '2025-04-30'},
    ]
    selected_resource = {'id': 'muestra-local', 'name': 'Muestra local de respaldo'}

raw_path = RAW_DIR / 'musica_colombia_raw.json'
raw_path.write_text(json.dumps(raw_rows, ensure_ascii=False, indent=2), encoding='utf-8')

print('Recurso seleccionado:', selected_resource.get('name'), selected_resource.get('id'))
print('Registros raw:', len(raw_rows))

## 3. Limpieza y organizacion

La limpieza convierte los datos heterogeneos en una estructura comun:

- Normalizacion de nombres de columnas.
- Estandarizacion de textos: espacios, mayusculas, acentos y valores vacios.
- Inferencia de campos clave: nombre, departamento, municipio, tipo y anio.
- Deduccion de tipo de actividad a partir de palabras como festival, concierto, formacion o convocatoria.
- Eliminacion de duplicados.
- Calculo de un puntaje de calidad por completitud.

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    text = str(value).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def normalize_key(key):
    key = normalize_text(key).lower()
    replacements = str.maketrans('áéíóúñ', 'aeioun')
    key = key.translate(replacements)
    key = re.sub(r'[^a-z0-9]+', '_', key).strip('_')
    return key

def pick(row, patterns):
    for column in row.index:
        if any(re.search(pattern, column) for pattern in patterns):
            return row[column]
    return ''

def infer_type(row):
    text = ' '.join(map(str, row.values)).lower()
    if re.search(r'festival|feria|festividad', text):
        return 'Festival'
    if re.search(r'concierto|recital|presentacion', text):
        return 'Concierto'
    if re.search(r'escuela|formacion|taller|clase', text):
        return 'Formacion'
    if re.search(r'convocatoria|beca|estimulo', text):
        return 'Convocatoria'
    if re.search(r'artista|agrupacion|banda|orquesta', text):
        return 'Agente musical'
    return normalize_text(pick(row, ['tipo', 'categoria', 'clase'])) or 'Actividad musical'

raw_df = pd.DataFrame(raw_rows)
raw_df.columns = [normalize_key(column) for column in raw_df.columns]

clean_records = []
for _, row in raw_df.iterrows():
    date_value = normalize_text(pick(row, ['fecha', 'anio', 'a_o', 'year', 'vigencia']))
    year_match = re.search(r'(20\d{2}|19\d{2})', date_value)
    record = {
        'nombre': normalize_text(pick(row, ['nombre', 'evento', 'actividad', 'titulo', 'razon_social', 'proyecto'])) or 'Actividad musical',
        'departamento': normalize_text(pick(row, ['departamento', 'depto', 'region'])) or 'Sin dato',
        'municipio': normalize_text(pick(row, ['municipio', 'ciudad', 'localidad'])) or 'Sin dato',
        'tipo': infer_type(row),
        'anio': int(year_match.group(1)) if year_match else None,
        'fuente': selected_resource.get('name'),
    }
    required = ['nombre', 'departamento', 'municipio', 'tipo', 'anio']
    record['calidad'] = round(sum(bool(record[field]) and record[field] != 'Sin dato' for field in required) / len(required) * 100, 2)
    clean_records.append(record)

clean_df = pd.DataFrame(clean_records).drop_duplicates()
clean_df.to_csv(PROCESSED_DIR / 'musica_colombia_limpio.csv', index=False, encoding='utf-8')
clean_df.head()

## 4. Movimiento de datos entre sistemas

Para demostrar transferencia, se mueve la informacion por tres capas:

1. **Archivo JSON raw:** conserva la extraccion original.
2. **Base SQLite staging:** simula una base operacional o zona intermedia.
3. **Data warehouse SQLite:** guarda la tabla limpia y agregados para BI.

En un entorno empresarial, estas capas podrian equivaler a un data lake en S3/ADLS, una base PostgreSQL staging y un warehouse como BigQuery, Snowflake o Redshift.

In [ ]:
staging_db = WAREHOUSE_DIR / 'staging_musica.db'
warehouse_db = WAREHOUSE_DIR / 'warehouse_musica.db'

with sqlite3.connect(staging_db) as conn:
    raw_df.to_sql('raw_musica', conn, if_exists='replace', index=False)

with sqlite3.connect(warehouse_db) as conn:
    clean_df.to_sql('fact_musica', conn, if_exists='replace', index=False)

print('Datos movidos a:', staging_db)
print('Datos publicados en:', warehouse_db)

## 5. Transformacion al formato requerido

El dashboard necesita datos resumidos. Por eso se crean tablas analiticas por departamento, tipo de actividad y calidad de datos. Estas tablas son mas livianas y responden rapido a preguntas de negocio.

In [ ]:
mart_departamento = clean_df.groupby('departamento', dropna=False).size().reset_index(name='registros').sort_values('registros', ascending=False)
mart_tipo = clean_df.groupby('tipo', dropna=False).size().reset_index(name='registros').sort_values('registros', ascending=False)
mart_calidad = pd.DataFrame({
    'metrica': ['registros', 'departamentos', 'municipios', 'calidad_promedio'],
    'valor': [len(clean_df), clean_df['departamento'].nunique(), clean_df['municipio'].nunique(), round(clean_df['calidad'].mean(), 2)]
})

with sqlite3.connect(warehouse_db) as conn:
    mart_departamento.to_sql('mart_departamento', conn, if_exists='replace', index=False)
    mart_tipo.to_sql('mart_tipo', conn, if_exists='replace', index=False)
    mart_calidad.to_sql('mart_calidad', conn, if_exists='replace', index=False)

mart_departamento.head(10)

## 6. Resultados finales

Los resultados se interpretan desde tres niveles:

- **Cobertura territorial:** departamentos y municipios presentes en el conjunto.
- **Actividad cultural:** categorias dominantes como festivales, conciertos, formacion o agentes musicales.
- **Calidad:** porcentaje de completitud de campos necesarios para analisis.

El dashboard ubicado en `dashboard/index.html` consume la misma idea de pipeline: busca datos, limpia, transforma y visualiza automaticamente.

In [ ]:
print('Resumen del pipeline')
print('Fuente:', selected_resource.get('name'))
print('Fecha de ejecucion:', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Registros limpios:', len(clean_df))
print('Departamentos:', clean_df['departamento'].nunique())
print('Municipios:', clean_df['municipio'].nunique())
print('Calidad promedio:', round(clean_df['calidad'].mean(), 2), '%')
print('\nTop tipos:')
print(mart_tipo.head(5).to_string(index=False))

## 7. Reflexion de aprendizaje

Este ejercicio muestra que un pipeline de datos no consiste solo en descargar archivos. Tambien requiere descubrir fuentes confiables, entender esquemas cambiantes, crear reglas de limpieza, mover datos entre capas y preparar salidas para usuarios finales. La parte mas importante fue convertir datos heterogeneos del catalogo publico en una estructura comun que un dashboard pueda analizar.

**Aprendizajes clave:**

- Las APIs abiertas facilitan automatizar la ingesta, pero los nombres de columnas pueden variar entre entidades.
- La calidad de datos debe medirse desde el inicio, no al final.
- Separar raw, limpio y mart permite trazabilidad y facilita depurar errores.
- Un dashboard es la capa de consumo, no reemplaza el pipeline; depende de una buena transformacion previa.
- Para produccion se agregarian orquestacion, logs, pruebas de calidad, versionamiento y monitoreo.